# 2. Fiche d'une unité

Pour une région, une préfecture, un canton, un site pilote ou le pays entier :

1. **Dernières observations** avec la normale du mois, l'écart, la part de la zone visible, la source et la nature (observé / dérivé).
2. **Série annuelle ou saisonnière** avec la moyenne de référence et la droite de Theil-Sen (pleine si la tendance est significative, pointillée sinon).
3. **24 derniers mois** comparés à la normale. Les mois couverts à moins de 50 % (nuages) sont en clair : ils valent `null` dans `data/api` (on affiche la valeur brute `value_raw`). Dans le livrable P2 du backend, ces mois sont publiés avec `low_coverage = true`, que l'interface doit signaler.
4. **Tableau des tendances** de l'unité, puis la **fiche JSON brute** (`data/api/summaries/<id>.json` ; les cantons n'en ont pas, leurs valeurs sont lues dans les tables).
5. La droite de Theil-Sen utilise la pente du tableau des tendances, sur la période testée ; son ordonnée est la médiane des résidus (affichage seulement).

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as w
from IPython.display import JSON, Markdown, display

import agbea_nb as nb

pd.set_option("display.max_rows", 60)
cat = nb.catalog()
print(f"Données : {nb.API}")
print(f"Générées le {cat['generated_at']} · pipeline {cat['pipeline_version']} · référence {cat['baseline']}")

In [ ]:
SERIES = ["rainfall", "ndvi", "lst_day", "lst_night", "air_temp", "rainfall_merra2"]


def observations(unit_id):
    s = nb.summary_or_none(unit_id)
    if s is None:  # cantons : pas de fiche JSON publiée, on lit les tables
        s = nb.summary_from_tables(unit_id)
        display(Markdown("*Pas de fiche JSON publiée pour ce niveau (cantons) : valeurs lues dans les tables.*"))
    u = s["unit"]
    e = s.get("elevation_m") or {}
    alt = f" · altitude moyenne {e['mean']:.0f} m (min {e['min']:.0f}, max {e['max']:.0f})" if e else ""
    display(Markdown(f"## {u['name']} — {nb.LEVEL_FR[u['level']]}\n"
                     f"`{u['id']}` · parent `{u['parent_id']}` · {u['area_km2']} km² · centre {u['centroid']}{alt}"))
    rows = []
    for ind, v in s["latest"].items():
        frac = v.get("valid_frac")
        rows.append({"indicateur": nb.label(ind), "période": v.get("date") or f"{v.get('start')} → {v.get('end')}",
                     "valeur": v.get("value"), "normale": v.get("clim_mean", v.get("normal")),
                     "écart": v.get("anomaly"), "écart %": v.get("anomaly_pct"), "part visible": frac,
                     "couverture < 50 %": (frac is not None and frac < 0.5), "source": v.get("dataset"),
                     "nature": v.get("nature")})
    display(pd.DataFrame(rows).round(3))


def serie(unit_id, ind, saison):
    g = nb.table("seasonal").query("unit_id == @unit_id and indicator == @ind and season == @saison").sort_values("year")
    if g.value.notna().sum() == 0:
        print("Pas de série pour ce choix.")
        return
    unit = cat["indicators"][ind]["unit"]
    fig = go.Figure(go.Bar(x=g.year, y=g.value, name="valeur", marker_color="#4a7bb7",
                           hovertemplate="%{x} : %{y:.3g} " + unit + "<extra></extra>"))
    base = g.baseline_mean.dropna()
    if len(base):
        fig.add_hline(y=base.iloc[0], line_dash="dot", line_color="#555",
                      annotation_text=f"référence {cat['baseline']} : {base.iloc[0]:.3g}")
    t = nb.table("trends").query("unit_id == @unit_id and indicator == @ind and season == @saison")
    txt = "pas assez d'années pour un test"
    if len(t):
        t = t.iloc[0]
        yrs = g.year.to_numpy(float)
        fit = nb.sen_line(yrs, g.value.to_numpy(float), t.sen_slope_per_year)
        span = (yrs >= t.first_year) & (yrs <= t.last_year)  # droite limitée à la période testée
        sig = bool(t.significant)
        fig.add_trace(go.Scatter(x=g.year[span], y=fit[span], mode="lines", name="Theil-Sen",
                                 line=dict(color="#c0392b" if sig else "#888", dash="solid" if sig else "dash")))
        txt = (f"{t.direction} de {t.sen_slope_per_decade:+.3g} {unit} / décennie "
               f"(IC 95 % {t.sen_ci95_low_per_decade:+.3g} à {t.sen_ci95_high_per_decade:+.3g}, p = {t.p_value:.3f}, "
               f"{t.first_year}-{t.last_year})")
        if not sig:
            txt = f"pas de tendance significative (pente {t.sen_slope_per_decade:+.3g} / décennie, p = {t.p_value:.3f})"
    fig.update_layout(title=f"{nb.unit_label(unit_id)} — {nb.label(ind)}, {nb.SEASON_FR[saison]}<br><sub>{txt}</sub>",
                      height=420, yaxis_title=unit, showlegend=False, margin=dict(t=80))
    fig.show()


def recents(unit_id, ind, n=24):
    m = nb.table("monthly").query("unit_id == @unit_id and indicator == @ind").sort_values("date").tail(n)
    if m.empty:
        print("Pas de données mensuelles.")
        return
    unit = cat["indicators"][ind]["unit"]
    low = m.valid_frac < 0.5
    fig = go.Figure()
    fig.add_trace(go.Bar(x=m.date[~low], y=m.value[~low], name="valeur publiée", marker_color="#4a7bb7"))
    if low.any():
        fig.add_trace(go.Bar(x=m.date[low], y=m.value_raw[low], name="couverture < 50 % : null dans data/api (valeur brute)",
                             marker_color="#c9d6ea", marker_line=dict(color="#4a7bb7", width=1)))
    fig.add_trace(go.Scatter(x=m.date, y=m.clim_mean, mode="lines+markers", name=f"normale {cat['baseline']}",
                             line=dict(color="#333", dash="dot")))
    fig.update_layout(title=f"{nb.unit_label(unit_id)} — {nb.label(ind)}, {n} derniers mois", height=380,
                      yaxis_title=unit, barmode="overlay", legend=dict(orientation="h", y=-0.2))
    fig.show()
    display(m[["date", "value", "value_raw", "clim_mean", "anomaly", "anomaly_pct", "valid_frac"]].round(3)
            .set_index("date").T)


def tendances(unit_id):
    t = nb.table("trends").query("unit_id == @unit_id")
    if t.empty:
        print("Aucune tendance calculée pour cette unité.")
        return
    t = t.assign(indicateur=t.indicator.map(nb.label), saison=t.season.map(nb.SEASON_FR))
    display(t[["indicateur", "saison", "first_year", "last_year", "n_years", "period_mean", "sen_slope_per_decade",
               "sen_ci95_low_per_decade", "sen_ci95_high_per_decade", "p_value", "significant", "direction"]]
            .round(4).reset_index(drop=True))

## Choisir une unité
Les graphiques et tableaux ci-dessous suivent ce choix.

In [ ]:
choix_unite = w.Dropdown(options=nb.unit_options(), value="lome", description="Unité", layout=w.Layout(width="480px"))
choix_ind = w.Dropdown(options=[(nb.label(i), i) for i in SERIES], value="rainfall", description="Indicateur")
choix_saison = w.Dropdown(options=[(v, k) for k, v in nb.SEASON_FR.items()], value="annual", description="Saison")


def tout(unit_id, ind, saison):
    observations(unit_id)
    serie(unit_id, ind, saison)
    recents(unit_id, ind)
    display(Markdown("### Toutes les tendances de l'unité"))
    tendances(unit_id)


sortie = w.interactive_output(tout, {"unit_id": choix_unite, "ind": choix_ind, "saison": choix_saison})
display(w.HBox([choix_unite, choix_ind, choix_saison]), sortie)

## Fiche JSON brute
Exactement ce que le backend ou le front peuvent lire (`data/api/summaries/`).

In [ ]:
_s = nb.summary_or_none(choix_unite.value)
JSON(_s, expanded=False) if _s else print("Pas de fiche JSON publiée pour cette unité (cantons).")